# Neural Architecture Search (NAS)

## 1. Search Space Design Example

In [1]:
import numpy as np

# Define NAS search space
class SearchSpace:
    def __init__(self):
        # Available operations
        self.operations = [
            'conv_3x3',
            'conv_5x5',
            'sep_conv_3x3',
            'sep_conv_5x5',
            'max_pool_3x3',
            'avg_pool_3x3',
            'identity',
            'zero'
        ]

        # Cell structure parameters
        self.num_nodes = 4  # Number of nodes in cell
        self.num_edges_per_node = 2  # Number of input edges per node

    def calculate_space_size(self):
        """Calculate search space size"""
        num_ops = len(self.operations)

        # Calculate choices for each node
        total_choices = 1
        for node_id in range(2, self.num_nodes + 2):
            # Choice of edge source (select from previous nodes)
            edge_choices = node_id
            # Choice of operation
            op_choices = num_ops
            # Choices for this node
            node_choices = (edge_choices * op_choices) ** self.num_edges_per_node
            total_choices *= node_choices

        return total_choices

    def sample_architecture(self):
        """Randomly sample an architecture"""
        architecture = []

        for node_id in range(2, self.num_nodes + 2):
            # Select inputs to this node
            node_config = []
            for _ in range(self.num_edges_per_node):
                # Input source node
                input_node = np.random.randint(0, node_id)
                # Operation
                operation = np.random.choice(self.operations)
                node_config.append((input_node, operation))

            architecture.append(node_config)

        return architecture

# Calculate search space size
search_space = SearchSpace()
space_size = search_space.calculate_space_size()

print("=== NAS Search Space Analysis ===")
print(f"Types of operations: {len(search_space.operations)}")
print(f"Number of nodes in cell: {search_space.num_nodes}")
print(f"Search space size: {space_size:,}")
print(f"Scientific notation: {space_size:.2e}")

# Sample architecture
sample = search_space.sample_architecture()
print(f"\n=== Sample Architecture ===")
for i, node in enumerate(sample, start=2):
    print(f"Node {i}:")
    for j, (input_node, op) in enumerate(node):
        print(f"  Input{j}: Node{input_node} → {op}")

=== NAS Search Space Analysis ===
Types of operations: 8
Number of nodes in cell: 4
Search space size: 241,591,910,400
Scientific notation: 2.42e+11

=== Sample Architecture ===
Node 2:
  Input0: Node0 → conv_5x5
  Input1: Node0 → sep_conv_3x3
Node 3:
  Input0: Node1 → zero
  Input1: Node1 → sep_conv_3x3
Node 4:
  Input0: Node0 → conv_3x3
  Input1: Node0 → zero
Node 5:
  Input0: Node4 → conv_3x3
  Input1: Node3 → conv_5x5


# 2. NAS Search Strategies

##  2.1. Reinforcement Learning-based (NASNet)

In [2]:
# NASNet-style reinforcement learning search (conceptual implementation)
import numpy as np

class RLController:
    """Reinforcement learning-based NAS controller"""

    def __init__(self, search_space):
        self.search_space = search_space
        self.history = []

    def sample_architecture(self, epsilon=0.1):
        """Sample architecture using epsilon-greedy strategy"""
        if np.random.random() < epsilon:
            # Exploration: random sampling
            return self.search_space.sample_architecture()
        else:
            # Exploitation: mutate from past good architectures
            if self.history:
                best_arch = max(self.history, key=lambda x: x['reward'])
                return self.mutate_architecture(best_arch['architecture'])
            else:
                return self.search_space.sample_architecture()

    def mutate_architecture(self, architecture):
        """Add small mutation to architecture"""
        mutated = [node[:] for node in architecture]

        # Randomly mutate one node
        node_idx = np.random.randint(len(mutated))
        edge_idx = np.random.randint(len(mutated[node_idx]))

        input_node, _ = mutated[node_idx][edge_idx]
        new_op = np.random.choice(self.search_space.operations)
        mutated[node_idx][edge_idx] = (input_node, new_op)

        return mutated

    def update(self, architecture, reward):
        """Receive reward and update history"""
        self.history.append({
            'architecture': architecture,
            'reward': reward
        })

# Simulation
search_space = SearchSpace()
controller = RLController(search_space)

print("=== Reinforcement Learning NAS Simulation ===")
for iteration in range(10):
    # Sample architecture
    arch = controller.sample_architecture(epsilon=0.3)

    # Simulate reward (in practice, train and obtain accuracy)
    # Here we use dummy reward based on operation diversity
    ops_used = set()
    for node in arch:
        for _, op in node:
            ops_used.add(op)
    reward = len(ops_used) / len(search_space.operations) + np.random.normal(0, 0.1)

    # Update controller
    controller.update(arch, reward)

    print(f"Iteration {iteration + 1}: Reward = {reward:.3f}")

# Display best architecture
best = max(controller.history, key=lambda x: x['reward'])
print(f"\n=== Best Architecture (Reward: {best['reward']:.3f}) ===")
for i, node in enumerate(best['architecture'], start=2):
    print(f"Node {i}:")
    for j, (input_node, op) in enumerate(node):
        print(f"  Input{j}: Node{input_node} → {op}")

=== Reinforcement Learning NAS Simulation ===
Iteration 1: Reward = 0.745
Iteration 2: Reward = 0.359
Iteration 3: Reward = 0.604
Iteration 4: Reward = 0.795
Iteration 5: Reward = 0.638
Iteration 6: Reward = 0.856
Iteration 7: Reward = 0.790
Iteration 8: Reward = 0.704
Iteration 9: Reward = 0.588
Iteration 10: Reward = 0.983

=== Best Architecture (Reward: 0.983) ===
Node 2:
  Input0: Node0 → zero
  Input1: Node1 → conv_3x3
Node 3:
  Input0: Node2 → sep_conv_5x5
  Input1: Node1 → conv_3x3
Node 4:
  Input0: Node2 → sep_conv_5x5
  Input1: Node2 → sep_conv_3x3
Node 5:
  Input0: Node0 → conv_5x5
  Input1: Node4 → max_pool_3x3


##  2.2. Evolutionary Algorithm

In [3]:
# Evolutionary algorithm NAS (simplified version)
import random
import copy

class EvolutionaryNAS:
    """Evolutionary algorithm-based NAS"""

    def __init__(self, search_space, population_size=20, num_generations=10):
        self.search_space = search_space
        self.population_size = population_size
        self.num_generations = num_generations

    def initialize_population(self):
        """Generate initial population"""
        return [self.search_space.sample_architecture()
                for _ in range(self.population_size)]

    def evaluate_fitness(self, architecture):
        """Evaluate fitness (dummy implementation)"""
        # In practice, train network and measure accuracy
        # Here we use operation diversity as score
        ops_used = set()
        for node in architecture:
            for _, op in node:
                ops_used.add(op)
        return len(ops_used) + random.gauss(0, 1)

    def select_parents(self, population, fitness_scores, k=2):
        """Tournament selection"""
        selected = []
        for _ in range(2):
            candidates_idx = random.sample(range(len(population)), k)
            best_idx = max(candidates_idx, key=lambda i: fitness_scores[i])
            selected.append(copy.deepcopy(population[best_idx]))
        return selected

    def crossover(self, parent1, parent2):
        """Crossover (one-point crossover)"""
        crossover_point = random.randint(1, len(parent1) - 1)
        child1 = parent1[:crossover_point] + parent2[crossover_point:]
        child2 = parent2[:crossover_point] + parent1[crossover_point:]
        return child1, child2

    def mutate(self, architecture, mutation_rate=0.1):
        """Mutation"""
        mutated = copy.deepcopy(architecture)

        for node_idx in range(len(mutated)):
            for edge_idx in range(len(mutated[node_idx])):
                if random.random() < mutation_rate:
                    input_node, _ = mutated[node_idx][edge_idx]
                    new_op = random.choice(self.search_space.operations)
                    mutated[node_idx][edge_idx] = (input_node, new_op)

        return mutated

    def run(self):
        """Run evolutionary algorithm"""
        # Initial population
        population = self.initialize_population()

        best_history = []

        for generation in range(self.num_generations):
            # Fitness evaluation
            fitness_scores = [self.evaluate_fitness(arch) for arch in population]

            # Statistics
            best_fitness = max(fitness_scores)
            avg_fitness = sum(fitness_scores) / len(fitness_scores)
            best_history.append(best_fitness)

            print(f"Generation {generation + 1}: Best={best_fitness:.3f}, Average={avg_fitness:.3f}")

            # Generate new generation
            new_population = []

            # Elitism
            elite_idx = fitness_scores.index(max(fitness_scores))
            new_population.append(copy.deepcopy(population[elite_idx]))

            # Selection, crossover, mutation
            while len(new_population) < self.population_size:
                parents = self.select_parents(population, fitness_scores)
                offspring1, offspring2 = self.crossover(parents[0], parents[1])
                offspring1 = self.mutate(offspring1)
                offspring2 = self.mutate(offspring2)

                new_population.extend([offspring1, offspring2])

            population = new_population[:self.population_size]

        # Return best individual
        fitness_scores = [self.evaluate_fitness(arch) for arch in population]
        best_idx = fitness_scores.index(max(fitness_scores))

        return population[best_idx], best_history

# Execute
search_space = SearchSpace()
evo_nas = EvolutionaryNAS(search_space, population_size=20, num_generations=10)

print("=== Evolutionary Algorithm NAS ===")
best_arch, history = evo_nas.run()

print(f"\n=== Best Architecture ===")
for i, node in enumerate(best_arch, start=2):
    print(f"Node {i}:")
    for j, (input_node, op) in enumerate(node):
        print(f"  Input{j}: Node{input_node} → {op}")

=== Evolutionary Algorithm NAS ===
Generation 1: Best=7.136, Average=4.961
Generation 2: Best=7.784, Average=5.399
Generation 3: Best=8.964, Average=5.996
Generation 4: Best=8.486, Average=5.344
Generation 5: Best=8.533, Average=6.211
Generation 6: Best=8.943, Average=6.035
Generation 7: Best=9.212, Average=6.280
Generation 8: Best=8.247, Average=5.490
Generation 9: Best=9.561, Average=5.719
Generation 10: Best=8.179, Average=5.400

=== Best Architecture ===
Node 2:
  Input0: Node0 → conv_5x5
  Input1: Node1 → sep_conv_5x5
Node 3:
  Input0: Node1 → identity
  Input1: Node0 → zero
Node 4:
  Input0: Node1 → conv_3x3
  Input1: Node3 → avg_pool_3x3
Node 5:
  Input0: Node1 → max_pool_3x3
  Input1: Node2 → avg_pool_3x3


## 2.3. DARTS (Differentiable Architecture Search)

DARTS Implementation (Simplified)

In [4]:
# Conceptual DARTS implementation (simplified version)
import torch
import torch.nn as nn
import torch.nn.functional as F

class MixedOp(nn.Module):
    """Weighted sum of multiple operations"""

    def __init__(self, C, stride):
        super(MixedOp, self).__init__()
        self._ops = nn.ModuleList()

        # Available operations
        self.operations = [
            ('sep_conv_3x3', lambda C, stride: SepConv(C, C, 3, stride, 1)),
            ('sep_conv_5x5', lambda C, stride: SepConv(C, C, 5, stride, 2)),
            ('avg_pool_3x3', lambda C, stride: nn.AvgPool2d(3, stride=stride, padding=1)),
            ('max_pool_3x3', lambda C, stride: nn.MaxPool2d(3, stride=stride, padding=1)),
            ('skip_connect', lambda C, stride: nn.Identity() if stride == 1 else FactorizedReduce(C, C)),
        ]

        for name, op in self.operations:
            self._ops.append(op(C, stride))

    def forward(self, x, weights):
        """Compute weighted sum"""
        return sum(w * op(x) for w, op in zip(weights, self._ops))

class Cell(nn.Module):
    """DARTS Cell"""

    def __init__(self, num_nodes, C_prev, C, reduction):
        super(Cell, self).__init__()
        self.num_nodes = num_nodes

        # Operation for each edge
        self._ops = nn.ModuleList()
        for i in range(num_nodes):
            for j in range(2 + i):
                stride = 2 if reduction and j < 2 else 1
                op = MixedOp(C, stride)
                self._ops.append(op)

    def forward(self, s0, s1, weights):
        """Forward propagation"""
        states = [s0, s1]
        offset = 0

        for i in range(self.num_nodes):
            s = sum(self._ops[offset + j](h, weights[offset + j])
                   for j, h in enumerate(states))
            offset += len(states)
            states.append(s)

        return torch.cat(states[-self.num_nodes:], dim=1)

class DARTSNetwork(nn.Module):
    """DARTS search network"""

    def __init__(self, C=16, num_cells=8, num_nodes=4, num_classes=10):
        super(DARTSNetwork, self).__init__()
        self.num_cells = num_cells
        self.num_nodes = num_nodes

        # Architecture parameters (α)
        num_ops = 5  # Types of operations
        num_edges = sum(2 + i for i in range(num_nodes))
        self.alphas_normal = nn.Parameter(torch.randn(num_edges, num_ops))
        self.alphas_reduce = nn.Parameter(torch.randn(num_edges, num_ops))

        # Network weights (w)
        self.stem = nn.Sequential(
            nn.Conv2d(3, C, 3, padding=1, bias=False),
            nn.BatchNorm2d(C)
        )

        # Cell construction is simplified
        self.cells = nn.ModuleList()
        # ... (in actual implementation, add multiple cells)

        self.classifier = nn.Linear(C, num_classes)

    def arch_parameters(self):
        """Return architecture parameters"""
        return [self.alphas_normal, self.alphas_reduce]

    def weights_parameters(self):
        """Return network weights"""
        return [p for n, p in self.named_parameters()
                if 'alpha' not in n]

# DARTS usage example
print("=== DARTS Conceptual Model ===")
model = DARTSNetwork(C=16, num_cells=8, num_nodes=4, num_classes=10)

print(f"Architecture parameter count: {sum(p.numel() for p in model.arch_parameters())}")
print(f"Network weight parameter count: {sum(p.numel() for p in model.weights_parameters())}")

# Architecture parameter shapes
print(f"\nNormal cell α: {model.alphas_normal.shape}")
print(f"Reduction cell α: {model.alphas_reduce.shape}")

# Normalize with softmax
weights_normal = F.softmax(model.alphas_normal, dim=-1)
print(f"\nNormalized weights (Normal cell, first edge):")
print(weights_normal[0].detach().numpy())

=== DARTS Conceptual Model ===
Architecture parameter count: 140
Network weight parameter count: 634

Normal cell α: torch.Size([14, 5])
Reduction cell α: torch.Size([14, 5])

Normalized weights (Normal cell, first edge):
[0.09679329 0.11697678 0.19816574 0.5843475  0.00371671]


DARTS Training Algorithm

In [5]:
# DARTS training procedure (pseudocode)
import torch
import torch.optim as optim

class DARTSTrainer:
    """DARTS training class"""

    def __init__(self, model, train_loader, val_loader):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader

        # Two optimizers
        self.optimizer_w = optim.SGD(
            model.weights_parameters(),
            lr=0.025,
            momentum=0.9,
            weight_decay=3e-4
        )

        self.optimizer_alpha = optim.Adam(
            model.arch_parameters(),
            lr=3e-4,
            betas=(0.5, 0.999),
            weight_decay=1e-3
        )

        self.criterion = nn.CrossEntropyLoss()

    def train_step(self, train_data, val_data):
        """One training step"""
        # 1. Update architecture parameters (α)
        self.model.train()
        x_val, y_val = val_data

        self.optimizer_alpha.zero_grad()
        logits = self.model(x_val)
        loss_alpha = self.criterion(logits, y_val)
        loss_alpha.backward()
        self.optimizer_alpha.step()

        # 2. Update network weights (w)
        x_train, y_train = train_data

        self.optimizer_w.zero_grad()
        logits = self.model(x_train)
        loss_w = self.criterion(logits, y_train)
        loss_w.backward()
        self.optimizer_w.step()

        return loss_w.item(), loss_alpha.item()

    def derive_architecture(self):
        """Derive final architecture"""
        # Select operation with highest weight for each edge
        def parse_alpha(alpha):
            gene = []
            n = 2
            start = 0
            for i in range(self.model.num_nodes):
                end = start + n
                W = alpha[start:end].copy()

                # Select two best operations for each edge
                edges = sorted(range(W.shape[0]),
                              key=lambda x: -max(W[x]))[:2]

                for j in edges:
                    k_best = W[j].argmax()
                    gene.append((j, k_best))

                start = end
                n += 1

            return gene

        # Normalize with softmax
        weights_normal = F.softmax(self.model.alphas_normal, dim=-1)
        weights_reduce = F.softmax(self.model.alphas_reduce, dim=-1)

        gene_normal = parse_alpha(weights_normal.data.cpu().numpy())
        gene_reduce = parse_alpha(weights_reduce.data.cpu().numpy())

        return gene_normal, gene_reduce

# Simulation example
print("=== DARTS Training Procedure ===")
print("1. Model initialization")
print("2. For each epoch:")
print("   a. Update α on validation data (architecture optimization)")
print("   b. Update w on training data (weight optimization)")
print("3. After training, select operation with highest weight for each edge")
print("4. Reconstruct network with selected operations and final training")

=== DARTS Training Procedure ===
1. Model initialization
2. For each epoch:
   a. Update α on validation data (architecture optimization)
   b. Update w on training data (weight optimization)
3. After training, select operation with highest weight for each edge
4. Reconstruct network with selected operations and final training


Practical DARTS Example (PyTorch)

In [6]:
# Example using actual DARTS implementation (using pt-darts library)
# Note: pt-darts is an external library (pip install pt-darts)

# Conceptual code example
"""
import torch
from darts import DARTS
from darts.api import spaces
from darts.trainer import DARTSTrainer

# Define search space
search_space = spaces.get_search_space('darts', 'cifar10')

# Build DARTS model
model = DARTS(
    C=16,
    num_classes=10,
    layers=8,
    criterion=nn.CrossEntropyLoss(),
    steps=4,
    multiplier=4,
    stem_multiplier=3
)

# Initialize trainer
trainer = DARTSTrainer(
    model,
    optimizer_config={
        'w_lr': 0.025,
        'w_momentum': 0.9,
        'w_weight_decay': 3e-4,
        'alpha_lr': 3e-4,
        'alpha_weight_decay': 1e-3
    }
)

# Run search
trainer.search(
    train_loader,
    val_loader,
    epochs=50
)

# Get best architecture
best_architecture = model.genotype()
print(f"Discovered architecture: {best_architecture}")
"""

print("=== Practical DARTS Usage ===")
print("1. Install libraries like pt-darts")
print("2. Define search space and model")
print("3. Search with bi-level optimization")
print("4. Retrain with discovered architecture")
print("\nDARTS advantages:")
print("- Search time: 4 GPU days (vs NASNet's 1800 GPU days)")
print("- High accuracy: 97%+ on CIFAR-10")
print("- Transferable: Applicable to ImageNet, etc.")

=== Practical DARTS Usage ===
1. Install libraries like pt-darts
2. Define search space and model
3. Search with bi-level optimization
4. Retrain with discovered architecture

DARTS advantages:
- Search time: 4 GPU days (vs NASNet's 1800 GPU days)
- High accuracy: 97%+ on CIFAR-10
- Transferable: Applicable to ImageNet, etc.


## 2.4. NAS Efficiency Techniques

 Weight Sharing (ENAS)

In [7]:
# Weight sharing concept (ENAS-style)
import torch
import torch.nn as nn

class SharedWeightSuperNet(nn.Module):
    """Weight-sharing supernet"""

    def __init__(self, num_nodes=4, C=16):
        super(SharedWeightSuperNet, self).__init__()
        self.num_nodes = num_nodes

        # Pre-construct all possible operations (shared weights)
        self.ops = nn.ModuleDict({
            'conv_3x3': nn.Conv2d(C, C, 3, padding=1),
            'conv_5x5': nn.Conv2d(C, C, 5, padding=2),
            'max_pool': nn.MaxPool2d(3, stride=1, padding=1),
            'avg_pool': nn.AvgPool2d(3, stride=1, padding=1),
            'identity': nn.Identity()
        })

    def forward(self, x, architecture):
        """
        architecture: Specifies operations for each node
        Example: [('conv_3x3', 0), ('max_pool', 1), ...]
        """
        states = [x, x]  # Initial states

        for node_id, (op_name, input_id) in enumerate(architecture):
            # Compute with specified operation and input
            s = self.ops[op_name](states[input_id])
            states.append(s)

        # Return output of last node
        return states[-1]

# Build supernet
supernet = SharedWeightSuperNet(num_nodes=4, C=16)

print("=== Weight Sharing (ENAS-style) ===")
print(f"Supernet parameter count: {sum(p.numel() for p in supernet.parameters()):,}")

# Use same weights for different architectures
arch1 = [('conv_3x3', 0), ('max_pool', 1), ('identity', 2), ('avg_pool', 1)]
arch2 = [('conv_5x5', 1), ('identity', 0), ('max_pool', 2), ('conv_3x3', 3)]

# Dummy input
x = torch.randn(1, 16, 32, 32)

output1 = supernet(x, arch1)
output2 = supernet(x, arch2)

print(f"\nArchitecture 1 output shape: {output1.shape}")
print(f"Architecture 2 output shape: {output2.shape}")
print("\n→ Can evaluate different architectures while sharing the same weights")

=== Weight Sharing (ENAS-style) ===
Supernet parameter count: 8,736

Architecture 1 output shape: torch.Size([1, 16, 32, 32])
Architecture 2 output shape: torch.Size([1, 16, 32, 32])

→ Can evaluate different architectures while sharing the same weights


NAS-Bench Dataset

In [8]:
class NASBenchSimulator:
    """NAS-Bench simulator"""

    def __init__(self):
        # Precomputed performance data (dummy)
        self.benchmark_data = {}
        self._populate_dummy_data()

    def _populate_dummy_data(self):
        """Generate dummy benchmark data"""
        import random
        random.seed(42)

        # Precompute performance for 100 architectures
        for i in range(100):
            arch_hash = f"arch_{i:03d}"
            self.benchmark_data[arch_hash] = {
                'val_accuracy': random.uniform(0.88, 0.95),
                'test_accuracy': random.uniform(0.87, 0.94),
                'training_time': random.uniform(100, 500),
                'params': random.randint(1_000_000, 10_000_000),
                'flops': random.randint(50_000_000, 500_000_000)
            }

    def query(self, architecture):
        """Query architecture performance (returns immediately)"""
        # Hash architecture
        arch_hash = self._hash_architecture(architecture)

        if arch_hash in self.benchmark_data:
            return self.benchmark_data[arch_hash]
        else:
            # Estimate unknown architecture
            return {
                'val_accuracy': 0.90,
                'test_accuracy': 0.89,
                'training_time': 300,
                'params': 5_000_000,
                'flops': 250_000_000
            }

    def _hash_architecture(self, architecture):
        """Hash architecture"""
        # Simple hash (actually more complex)
        arch_str = str(architecture)
        hash_val = sum(ord(c) for c in arch_str) % 100
        return f"arch_{hash_val:03d}"

# Use NAS-Bench
bench = NASBenchSimulator()

print("=== Fast Evaluation with NAS-Bench ===")

# Architecture search
import time

architectures = [
    [('conv_3x3', 0), ('max_pool', 1)],
    [('conv_5x5', 0), ('identity', 1)],
    [('avg_pool', 0), ('conv_3x3', 1)]
]

start_time = time.time()
results = []

for arch in architectures:
    result = bench.query(arch)
    results.append((arch, result))

end_time = time.time()

print(f"Search time: {end_time - start_time:.4f} seconds")
print(f"\n=== Search Results ===")
for arch, result in results:
    print(f"Architecture: {arch}")
    print(f"  Validation accuracy: {result['val_accuracy']:.3f}")
    print(f"  Test accuracy: {result['test_accuracy']:.3f}")
    print(f"  Training time: {result['training_time']:.1f} seconds")
    print(f"  Parameters: {result['params']:,}")
    print()

print("→ Can obtain performance immediately without actual training")

=== Fast Evaluation with NAS-Bench ===
Search time: 0.0003 seconds

=== Search Results ===
Architecture: [('conv_3x3', 0), ('max_pool', 1)]
  Validation accuracy: 0.895
  Test accuracy: 0.898
  Training time: 123.5 seconds
  Parameters: 7,358,113

Architecture: [('conv_5x5', 0), ('identity', 1)]
  Validation accuracy: 0.919
  Test accuracy: 0.889
  Training time: 341.9 seconds
  Parameters: 6,752,576

Architecture: [('avg_pool', 0), ('conv_3x3', 1)]
  Validation accuracy: 0.899
  Test accuracy: 0.904
  Training time: 315.7 seconds
  Parameters: 8,930,103

→ Can obtain performance immediately without actual training


Combining Efficiency Techniques

In [9]:
# Search combining multiple efficiency techniques
import numpy as np

class EfficientNAS:
    """Efficient NAS"""

    def __init__(self, use_weight_sharing=True, use_proxy=True,
                 use_early_stopping=True):
        self.use_weight_sharing = use_weight_sharing
        self.use_proxy = use_proxy
        self.use_early_stopping = use_early_stopping

        if use_weight_sharing:
            self.supernet = SharedWeightSuperNet()

        if use_proxy:
            self.proxy_epochs = 10  # 10 epochs instead of full training
            self.proxy_data_fraction = 0.2  # Use only 20% of data

    def evaluate_architecture(self, architecture, full_evaluation=False):
        """Evaluate architecture"""
        if full_evaluation:
            # Full evaluation (only for final candidates)
            epochs = 50
            data_fraction = 1.0
        else:
            # Proxy evaluation
            epochs = self.proxy_epochs if self.use_proxy else 50
            data_fraction = self.proxy_data_fraction if self.use_proxy else 1.0

        # Early stopping simulation
        if self.use_early_stopping:
            # Terminate if performance is poor in early epochs
            early_acc = np.random.random()
            if early_acc < 0.5:  # Threshold
                return {'accuracy': early_acc, 'stopped_early': True}

        # Evaluation (dummy)
        accuracy = np.random.uniform(0.85, 0.95)

        return {
            'accuracy': accuracy,
            'epochs': epochs,
            'data_fraction': data_fraction,
            'stopped_early': False
        }

    def search(self, num_candidates=100, top_k=5):
        """Run NAS search"""
        print("=== Efficient NAS Search ===")
        print(f"Weight Sharing: {self.use_weight_sharing}")
        print(f"Proxy Tasks: {self.use_proxy}")
        print(f"Early Stopping: {self.use_early_stopping}")
        print()

        candidates = []

        # 1. Large-scale proxy evaluation
        for i in range(num_candidates):
            arch = [('conv_3x3', 0), ('max_pool', 1)]  # Dummy
            result = self.evaluate_architecture(arch, full_evaluation=False)
            candidates.append((arch, result))

        # Candidates not terminated by early stopping
        valid_candidates = [c for c in candidates if not c[1]['stopped_early']]

        print(f"Initial candidates: {num_candidates}")
        print(f"Reduced by early stopping: {num_candidates - len(valid_candidates)}")

        # 2. Full evaluation of top K
        top_candidates = sorted(valid_candidates,
                               key=lambda x: x[1]['accuracy'],
                               reverse=True)[:top_k]

        print(f"Candidates for full evaluation: {top_k}")
        print()

        final_results = []
        for arch, proxy_result in top_candidates:
            full_result = self.evaluate_architecture(arch, full_evaluation=True)
            final_results.append((arch, full_result))

        # Return best candidate
        best = max(final_results, key=lambda x: x[1]['accuracy'])

        return best, final_results

# Execute
nas = EfficientNAS(
    use_weight_sharing=True,
    use_proxy=True,
    use_early_stopping=True
)

best_arch, all_results = nas.search(num_candidates=100, top_k=5)

print("=== Search Results ===")
print(f"Best architecture: {best_arch[0]}")
print(f"Best accuracy: {best_arch[1]['accuracy']:.3f}")
print(f"\nTop 5 accuracies:")
for i, (arch, result) in enumerate(all_results, 1):
    print(f"{i}. Accuracy={result['accuracy']:.3f}")

=== Efficient NAS Search ===
Weight Sharing: True
Proxy Tasks: True
Early Stopping: True

Initial candidates: 100
Reduced by early stopping: 51
Candidates for full evaluation: 5

=== Search Results ===
Best architecture: [('conv_3x3', 0), ('max_pool', 1)]
Best accuracy: 0.915

Top 5 accuracies:
1. Accuracy=0.868
2. Accuracy=0.162
3. Accuracy=0.881
4. Accuracy=0.915
5. Accuracy=0.064
